## Описание
Этот код для демонстрации другого подхода. По сути реализовать его не удалось, так как сильно не хватает видеопамяти. Суть такая - я нашёл gemma-1b обученную на математических задачах.
Соответственно возможно она могла бы лучше справиться с задачей. Проверить я этого так и не смог, так как нет времени и ресурсов на обучение

In [2]:
!unzip /content/gemma3-countdown.zip

Archive:  /content/gemma3-countdown.zip
  inflating: gemma3-countdown/adapter_model.safetensors  
  inflating: gemma3-countdown/adapter_config.json  
  inflating: gemma3-countdown/README.md  
  inflating: gemma3-countdown/training_args.bin  
  inflating: gemma3-countdown/chat_template.jinja  
  inflating: gemma3-countdown/tokenizer.json  
  inflating: gemma3-countdown/tokenizer_config.json  
  inflating: gemma3-countdown/checkpoint-1884/adapter_model.safetensors  
  inflating: gemma3-countdown/checkpoint-1884/adapter_config.json  
  inflating: gemma3-countdown/checkpoint-1884/rng_state.pth  
  inflating: gemma3-countdown/checkpoint-1884/optimizer.pt  
  inflating: gemma3-countdown/checkpoint-1884/README.md  
  inflating: gemma3-countdown/checkpoint-1884/scheduler.pt  
  inflating: gemma3-countdown/checkpoint-1884/training_args.bin  
  inflating: gemma3-countdown/checkpoint-1884/chat_template.jinja  
  inflating: gemma3-countdown/checkpoint-1884/tokenizer.json  
  inflating: gemma3-coun

In [1]:
!pip install -U torchao

In [2]:
!pip uninstall -y pyarrow
!pip install pyarrow==15.0.2

Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which i

In [2]:
# %%
# %%capture
!pip install -q -U \
    "transformers>=4.40.0" \
    "datasets>=2.19.0" \
    "trl>=0.8.6" \
    "peft>=0.10.0" \
    "accelerate>=0.29.0" \
    "bitsandbytes>=0.43.0" \
    "sentencepiece" \
    "protobuf"


In [ ]:
# %%
import os, re, ast, json, warnings
import pandas as pd
import numpy as np
import torch
warnings.filterwarnings("ignore")

# ── Пути ──────────────────────────────────────────────────────────────────────
HF_TOKEN = "1984_1984_1984"
MODEL_NAME      = "google/gemma-3-1b-it"
DATASET_NAME    = "HuggingFaceTB/Countdown-Task-GOLD"
DATASET_CONFIG  = "verified_Qwen2.5-7B-Instruct"
TEST_CSV        = "/content/test_public.csv"
OUTPUT_DIR      = "/kaggle/working/gemma3-countdown"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# ── Гиперпараметры ─────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH  = 256
LORA_RANK       = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
NUM_EPOCHS      = 2
LR              = 2e-4
WARMUP_RATIO    = 0.03
MAX_TRAIN_STEPS = None

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


CUDA: True
GPU:  Tesla T4
VRAM: 15.6 GB


In [2]:
# %%
from datasets import load_dataset

# Загружаем датасет учителя
raw_ds = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split="train",
    token=HF_TOKEN if HF_TOKEN else None,
)
print(f"Размер датасета: {len(raw_ds)} примеров")
print(f"Колонки: {raw_ds.column_names}")

# Смотрим на один пример
sample = raw_ds[0]["messages"]
if isinstance(sample, str):
    sample = json.loads(sample)
for msg in sample:
    print(f"\n[{msg['role']}]")
    print(msg["content"][:300])


Размер датасета: 30441 примеров
Колонки: ['target', 'nums', 'messages']

[system]
You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer.

[user]
Using the numbers [78, 46, 93], create an equation that equals 61. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 

[assistant]
<think>
To solve this, I need to use the numbers 78, 46, and 93 to create an equation that equals 61. I'll start by considering the basic arithmetic operations and how they can be used to get close to 61.

First, I'll try subtraction, as it's the most straightforward operation to get close to 61:
- 


In [3]:
def parse_prompt_column(raw_messages):
    """
    Принимает значение колонки prompt (list[dict] или JSON-строка).
    Возвращает (user_content: str, answer: str).
    """
    # Десериализуем если строка
    if isinstance(raw_messages, str):
        try:
            messages = json.loads(raw_messages)
        except json.JSONDecodeError:
            messages = ast.literal_eval(raw_messages)
    else:
        messages = raw_messages  # уже list

    user_content      = None
    assistant_content = None

    for msg in messages:
        role = msg.get("role", "")
        if role == "user":
            user_content = msg.get("content", "")
        elif role == "assistant":
            assistant_content = msg.get("content", "")

    # Извлекаем финальный ответ из тега <answer>
    if assistant_content:
        m = re.search(r"<answer>(.*?)</answer>", assistant_content, re.DOTALL)
        answer = m.group(1).strip() if m else assistant_content.strip()
    else:
        answer = ""

    return user_content or "", answer


# Быстрая проверка
u, a = parse_prompt_column(raw_ds[0]["messages"])
print("USER :", u[:120])
print("ANSWER:", a)


USER : Using the numbers [78, 46, 93], create an equation that equals 61. You can use basic arithmetic operations (+, -, *, /) 
ANSWER: 93 - (78 - 46) = 61


In [4]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN if HF_TOKEN else None,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"BOS token      : {tokenizer.bos_token!r}")
print(f"EOS token      : {tokenizer.eos_token!r}")
print(f"Chat template  :\n{tokenizer.chat_template[:300] if tokenizer.chat_template else 'None'}")


Vocab size     : 262144
BOS token      : '<bos>'
EOS token      : '<eos>'
Chat template  :
{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%-


In [5]:

#  промпт для студента (короткий, без упоминания think)
SYSTEM_PROMPT = (
    "You are a math puzzle solver. "
    "Given a target number and a list of numbers, "
    "find an arithmetic equation using those numbers that equals the target. "
    "Reply ONLY with: <answer>equation</answer>"
)

def format_example(example):
    """
    Переводим один обучающий пример в строку для Gemma-3.
    Формат: system+user → assistant с <answer>...</answer>.
    """
    user_content, answer = parse_prompt_column(example["messages"])

    if not user_content or not answer:
        return {"text": ""}

    messages = [
        {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\n{user_content}"},
        {"role": "assistant", "content": f"<answer>{answer}</answer>"},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


# Применяем к датасету
formatted_ds = raw_ds.map(
    format_example,
    remove_columns=raw_ds.column_names,
    num_proc=2,
    desc="Форматирование примеров",
)

# Убираем пустые строки
formatted_ds = formatted_ds.filter(lambda x: len(x["text"]) > 10)
print(f"Примеров после фильтрации: {len(formatted_ds)}")

# Смотрим первый пример
print("\n" + "="*60)
print(formatted_ds[0]["text"])


Примеров после фильтрации: 30441

<bos><start_of_turn>user
You are a math puzzle solver. Given a target number and a list of numbers, find an arithmetic equation using those numbers that equals the target. Reply ONLY with: <answer>equation</answer>

Using the numbers [78, 46, 93], create an equation that equals 61. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.<end_of_turn>
<start_of_turn>model
<answer>93 - (78 - 46) = 61</answer><end_of_turn>



In [6]:
# Разбивка на train / eval (1 % на валидацию)
split = formatted_ds.train_test_split(test_size=0.01, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")


Train: 30136 | Eval: 305


In [1]:
# Без этого не работает

import os
import shutil

# 1. Отключаем wandb полностью (раз ты и так используешь report_to="none")
os.environ["WANDB_DISABLED"] = "true"

# 2. Заставляем protobuf работать в чистом Python-режиме (это главное)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 3. Жёстко удаляем все старые сломанные protobuf-файлы wandb
wandb_dir = "/usr/local/lib/python3.12/dist-packages/wandb"
if os.path.exists(wandb_dir):
    shutil.rmtree(wandb_dir, ignore_errors=True)

print("Старые файлы wandb удалены + wandb отключён")

Старые файлы wandb удалены + wandb отключён


# Мат версия

In [ ]:
!zip -r /content/gemma3-countdown-contin.zip /content/gemma3-countdown-contin

  adding: content/gemma3-countdown-contin/ (stored 0%)
  adding: content/gemma3-countdown-contin/checkpoint-500/ (stored 0%)
  adding: content/gemma3-countdown-contin/checkpoint-500/trainer_state.json (deflated 69%)
  adding: content/gemma3-countdown-contin/checkpoint-500/scheduler.pt (deflated 61%)
  adding: content/gemma3-countdown-contin/checkpoint-500/optimizer.pt (deflated 10%)
  adding: content/gemma3-countdown-contin/checkpoint-500/training_args.bin (deflated 52%)
  adding: content/gemma3-countdown-contin/checkpoint-500/README.md (deflated 66%)
  adding: content/gemma3-countdown-contin/checkpoint-500/tokenizer.json (deflated 83%)
  adding: content/gemma3-countdown-contin/checkpoint-500/chat_template.jinja (deflated 70%)
  adding: content/gemma3-countdown-contin/checkpoint-500/tokenizer_config.json (deflated 60%)
  adding: content/gemma3-countdown-contin/checkpoint-500/adapter_model.safetensors (deflated 21%)
  adding: content/gemma3-countdown-contin/checkpoint-500/adapter_config

In [8]:
# %%
import time
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

# ---------------------------
# Параметры (адаптированные под math‑модель)
# ---------------------------
MODEL_NAME = "twm-opensource/gemma-3-1b-it"   # Математический файнтюн Gemma 1B
OUTPUT_DIR = "/content/gemma3-countdown-contin"

# Гиперпараметры
BATCH_SIZE = 16
GRAD_ACCUM = 2
MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 1
LR = 1e-4
WARMUP_RATIO = 0.05
MAX_TRAIN_STEPS = -1

# LoRA параметры
LORA_RANK = 100
LORA_ALPHA = 32
LORA_DROPOUT = 0

# ---------------------------
# 1. Загрузка модели с 4‑bit квантизацией
# ---------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# ---------------------------
# 2. Токенизатор
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

# ---------------------------
# 3. LoRA конфигурация
# ---------------------------
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------------------------
# 4. Аргументы тренировки
# ---------------------------
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_TRAIN_STEPS if MAX_TRAIN_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=int(WARMUP_RATIO * (len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))),
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=2,
    seed=42,
)

# ---------------------------
# 5. Тренер
# ---------------------------
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# ---------------------------
# 6. Запуск
# ---------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
model = model.to("cuda:0")
if hasattr(model, "base_model"):
    model.base_model = model.base_model.to("cuda:0")

print(f"Начинаем обучение {MODEL_NAME}...")
print(f"Шагов в эпохе: {len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)}")

start_time = time.time()
trainer.train()
elapsed = (time.time() - start_time) / 60
print(f"\n✅ Обучение завершено за {elapsed:.1f} минут")

# Сохраняем
trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final"))

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

trainable params: 81,536,000 || all params: 1,081,421,952 || trainable%: 7.5397


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Начинаем обучение twm-opensource/gemma-3-1b-it...
Шагов в эпохе: 941


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
500,0.151301,0.152152


Exception in thread Thread-5 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_s

KeyboardInterrupt: 

In [11]:
!zip -r gemma3-countdown-contin.zip /content/gemma3-countdown-contin

  adding: content/gemma3-countdown-contin/ (stored 0%)
  adding: content/gemma3-countdown-contin/checkpoint-500/ (stored 0%)
  adding: content/gemma3-countdown-contin/checkpoint-500/trainer_state.json (deflated 69%)
  adding: content/gemma3-countdown-contin/checkpoint-500/scheduler.pt (deflated 61%)
  adding: content/gemma3-countdown-contin/checkpoint-500/optimizer.pt (deflated 10%)
  adding: content/gemma3-countdown-contin/checkpoint-500/training_args.bin (deflated 53%)
  adding: content/gemma3-countdown-contin/checkpoint-500/README.md (deflated 66%)
  adding: content/gemma3-countdown-contin/checkpoint-500/tokenizer.json (deflated 83%)
  adding: content/gemma3-countdown-contin/checkpoint-500/chat_template.jinja (deflated 70%)
  adding: content/gemma3-countdown-contin/checkpoint-500/tokenizer_config.json (deflated 60%)
  adding: content/gemma3-countdown-contin/checkpoint-500/adapter_model.safetensors (deflated 22%)
  adding: content/gemma3-countdown-contin/checkpoint-500/adapter_config

In [13]:
import os
import time
import re
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel, PeftConfig

# ---------------------------
# 1. Параметры
# ---------------------------
CHECKPOINT_PATH = "/content/gemma3-countdown-contin/checkpoint-500"
TEST_CSV_PATH = "/content/test_public.csv"
MAX_NEW_TOKENS = 1024
BATCH_SIZE = 16

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---------------------------
# 2. Загрузка модели
# ---------------------------
print(f"Загружаем чекпоинт: {CHECKPOINT_PATH}")

peft_config = PeftConfig.from_pretrained(CHECKPOINT_PATH)
base_model_name = peft_config.base_model_name_or_path

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
model.eval()
print(f"Модель на устройстве: {next(model.parameters()).device}")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

print("✅ Модель и токенизатор загружены")

# ---------------------------
# 3. Промпт как в обучении
# ---------------------------
SYSTEM_PROMPT = (
    "You are a math puzzle solver. Given a target number and a list of numbers, "
    "find an arithmetic equation using those numbers that equals the target. "
    "Reply ONLY with: <answer>equation</answer>"
)

def build_test_prompt(target: int, nums_str: str) -> str:
    import ast
    try:
        nums = ast.literal_eval(nums_str)
    except:
        nums = [int(x) for x in re.findall(r'\d+', nums_str)]

    nums_display = ", ".join(str(n) for n in nums)

    user_content = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Using the numbers [{nums_display}], create an equation that equals {target}. "
        f"You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. "
        f"Show your work in <think> </think> tags. "
        f"And return the final equation and answer in <answer> </answer> tags, "
        f"for example <answer> (1 + 2) / 3 = 1 </answer>."
    )

    messages = [{"role": "user", "content": user_content}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt

# ---------------------------
# 4. Извлечение ответа из <answer>
# ---------------------------
def extract_equation(text: str) -> str:
    """Извлекает уравнение из <answer>...</answer>"""
    if not text:
        return "0"

    text = text.strip()

    # Обрезаем по <end_of_turn>
    if "<end_of_turn>" in text:
        text = text.split("<end_of_turn>")[0]

    # Ищем <answer> теги
    m = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if m:
        answer_content = m.group(1).strip()

        # Извлекаем только выражение (до "=")
        if "=" in answer_content:
            expr = answer_content.split("=")[0].strip()
        else:
            expr = answer_content

        # Очищаем от лишних символов
        expr = re.sub(r'[^\d\s\+\-\*\/\(\)\.]', '', expr)
        expr = re.sub(r'\s+', '', expr)

        if expr and re.search(r'[\+\-\*\/]', expr):
            return expr

    # Fallback: ищем любое выражение
    expr_match = re.search(r'[\d\s\+\-\*\/\(\)]+', text.replace(' ', ''))
    if expr_match:
        expr = expr_match.group()
        if re.search(r'[\+\-\*\/]', expr):
            return expr

    return "0"

# ---------------------------
# 5. генерация
# ---------------------------
def batch_generate(prompts, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    all_answers = []

    device = next(model.parameters()).device
    print(f"Генерация на устройстве: {device}")

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                stop_strings=["<end_of_turn>"],
                tokenizer=tokenizer,
            )

        input_len = enc["input_ids"].shape[1]
        for seq in gen_ids:
            new_tokens = seq[input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            answer = extract_equation(text)
            all_answers.append(answer)

        # Прогресс
        if (i // batch_size) % 10 == 0:
            processed = min(i+batch_size, len(prompts))
            print(f"  Обработано {processed}/{len(prompts)} ({100*processed/len(prompts):.1f}%)")
            torch.cuda.empty_cache()

    return all_answers

# ---------------------------
# 6. Загрузка и тестирование
# ---------------------------
print("Загружаем тестовый датасет...")
test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Тестовых примеров: {len(test_df)}")

# Тестируем на 5 примерах
print("\n🔍 Тестирование на 5 примерах:")
print("="*70)

for i in range(min(5, len(test_df))):
    row = test_df.iloc[i]
    prompt = build_test_prompt(row["target"], row["nums"])

    print(f"\n--- Пример {i+1} (ID: {row['id']}) ---")
    print(f"Target: {row['target']}, Nums: {row['nums']}")

    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            stop_strings=["<end_of_turn>"],
            tokenizer=tokenizer
        )

    output = tokenizer.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

    # Показываем вывод
    print(f"Raw output: {output[:200]}...")

    # Извлекаем уравнение
    equation = extract_equation(output)
    print(f"✅ Extracted equation: {equation}")

print("\n" + "="*70)

# ---------------------------
# 7. Полная генерация
# ---------------------------
print("Формируем все промпты...")
test_prompts = [
    build_test_prompt(row["target"], row["nums"])
    for _, row in test_df.iterrows()
]

print("Генерируем ответы...")
start = time.time()

dummy_input = tokenizer(["test"], return_tensors="pt").to(model.device)
with torch.no_grad():
    _ = model.generate(**dummy_input, max_new_tokens=1)

raw_answers = batch_generate(test_prompts)
elapsed = (time.time() - start) / 60
print(f"✅ Генерация завершена за {elapsed:.1f} минут ({len(test_prompts)/elapsed:.0f} примеров/мин)")

# ---------------------------
# 8. Сохранение результатов
# ---------------------------
submission = pd.DataFrame({
    "id": test_df["id"],
    "equation": raw_answers
})

submission["equation"] = submission["equation"].fillna("0").replace("", "0")

submission_path = "submission_trained_fixed_prompt.csv"
submission.to_csv(submission_path, index=False)
print(f"\n📁 Submission сохранён: {submission_path}")

# Статистика
non_zero = (submission["equation"] != "0").sum()
print(f"\n📊 Статистика:")
print(f"  Всего: {len(submission)}")
print(f"  Найдено выражений: {non_zero} ({non_zero/len(submission)*100:.1f}%)")
print(f"  Пустых (0): {len(submission)-non_zero}")

print(f"\n📝 Первые 10 ответов:")
for i in range(min(10, len(submission))):
    row = submission.iloc[i]
    test_row = test_df[test_df['id'] == row['id']].iloc[0]
    print(f"  ID {row['id']}: {test_row['target']} = {row['equation']}")

Загружаем чекпоинт: /content/gemma3-countdown-contin/checkpoint-500


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Модель на устройстве: cuda:0
✅ Модель и токенизатор загружены
Загружаем тестовый датасет...
Тестовых примеров: 2000

🔍 Тестирование на 5 примерах:

--- Пример 1 (ID: 0) ---
Target: 870, Nums: [11, 83, 62, 23, 68]
Raw output: <answer>83 + 68 + 23 + 11 + 62 = 870</answer>...
✅ Extracted equation: 83+68+23+11+62

--- Пример 2 (ID: 1) ---
Target: 705, Nums: [71, 66, 9]
Raw output: <answer>71 + 66 - 9 = 705</answer>...
✅ Extracted equation: 71+66-9

--- Пример 3 (ID: 2) ---
Target: 763, Nums: [82, 51, 59, 46]
Raw output: <answer>82 + 51 - 59 + 46 = 763</answer>...
✅ Extracted equation: 82+51-59+46

--- Пример 4 (ID: 3) ---
Target: 403, Nums: [59, 52, 89, 10, 97, 96]
Raw output: <answer>96 - 59 + 52 + 97 - 10 = 403</answer>...
✅ Extracted equation: 96-59+52+97-10

--- Пример 5 (ID: 4) ---
Target: 67, Nums: [57, 74, 98, 20, 34]
Raw output: <answer>57 + 74 - 98 + 20 + 34 = 67</answer>...
✅ Extracted equation: 57+74-98+20+34

Формируем все промпты...
Генерируем ответы...
Генерация на устройстве

KeyboardInterrupt: 